# TA-Lib Factor Screen (IC / ICIR)

Quick screen of store TA-Lib factors (`rsi`, `adx`, `mfi`, `bb_percent_b`) on the S1 **trade-date** train panel.

- No hypothesis ID — lives under `other_tests/`
- Alphalens periods `(1, 5, 21)`; **prioritise 5d** when ranking
- Print **mean IC** and **ICIR** only — **no** full tear sheets / PDFs
- Pivot prices = `open` (no `shift(-1)`)
- Do not enforce IC thresholds in code; report values only


## 0. Imports & Config


In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import alphalens as al

from data.processing.s1_feature_store import add_talib_factors

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)

PERIODS = (1, 5, 21)
PRIMARY_PERIOD = 5

# Screen grids (classic defaults + short/medium neighbours; nbdev fixed)
RSI_PERIODS = [2, 5, 7, 10, 14, 21, 28]
ADX_PERIODS = [7, 10, 14, 20, 28]
MFI_PERIODS = [7, 10, 14, 20, 28]
BB_PERIODS = [10, 15, 20, 30, 40, 50]
BB_NBDEV = 2.0

warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Data Loading


In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
panel["date"] = pd.to_datetime(panel["date"])
print(panel.shape)
print(panel["date"].min(), "→", panel["date"].max())
print("tickers:", panel["ticker"].nunique())


(289381, 11)
2010-01-05 00:00:00 → 2021-08-03 00:00:00
tickers: 100


## 2. Data Cleaning & Engineering

No winsorize / floor in the store. Drop rows missing OHLCV needed by TA-Lib.


In [3]:
ohlcv_cols = ["open", "high", "low", "close", "volume"]
work = panel.dropna(subset=ohlcv_cols).copy()
print("rows after OHLCV dropna:", len(work))


rows after OHLCV dropna: 289381


## 3. Modeling / Signal Construction

Build all screened TA-Lib columns via `add_talib_factors`.


In [4]:
work = add_talib_factors(
    work,
    feature_subset=["rsi"],
    timeperiod=RSI_PERIODS,
)
work = add_talib_factors(
    work,
    feature_subset=["adx"],
    timeperiod=ADX_PERIODS,
)
work = add_talib_factors(
    work,
    feature_subset=["mfi"],
    timeperiod=MFI_PERIODS,
)
work = add_talib_factors(
    work,
    feature_subset=["bb_percent_b"],
    bb_timeperiod=BB_PERIODS,
    bb_nbdev=BB_NBDEV,
)

factor_cols = [
    c
    for c in work.columns
    if c.startswith(("rsi_", "adx_", "mfi_", "bb_percent_b_"))
    or c in {"rsi", "adx", "mfi", "bb_percent_b"}
]
print("factor columns:", len(factor_cols))
print(sorted(factor_cols)[:12], "...")


factor columns: 23
['adx_10', 'adx_14', 'adx_20', 'adx_28', 'adx_7', 'bb_percent_b_10', 'bb_percent_b_15', 'bb_percent_b_20', 'bb_percent_b_30', 'bb_percent_b_40', 'bb_percent_b_50', 'mfi_10'] ...


## 4. Evaluation

Mean Spearman IC and ICIR (`mean(IC) / std(IC)`) at 1d / 5d / 21d. Sorted by **|IC_5d|** then signed IC_5d tables.


In [5]:
def ic_icir_table(df: pd.DataFrame, factor_col: str, periods=PERIODS) -> dict:
    factor = (
        df[["date", "ticker", factor_col]]
        .dropna(subset=[factor_col])
        .set_index(["date", "ticker"])[factor_col]
    )
    prices = (
        df.pivot(index="date", columns="ticker", values="open")
        .sort_index()
    )
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor,
        prices,
        periods=periods,
        quantiles=5,
        max_loss=0.35,
    )
    ic = al.performance.factor_information_coefficient(factor_data)
    out = {"factor": factor_col}
    for p in periods:
        col = f"{p}D"
        if col not in ic.columns:
            # alphalens may label as '{p}D' or period int depending on version
            matches = [c for c in ic.columns if str(p) in str(c)]
            col = matches[0] if matches else ic.columns[periods.index(p)]
        series = ic[col].dropna()
        mean_ic = float(series.mean()) if len(series) else np.nan
        std_ic = float(series.std(ddof=1)) if len(series) > 1 else np.nan
        icir = mean_ic / std_ic if std_ic and np.isfinite(std_ic) and std_ic != 0 else np.nan
        out[f"ic_{p}d"] = mean_ic
        out[f"icir_{p}d"] = icir
    return out


rows = []
for col in sorted(factor_cols):
    try:
        rows.append(ic_icir_table(work, col))
        print("ok", col)
    except Exception as exc:
        print("FAIL", col, type(exc).__name__, exc)

results = pd.DataFrame(rows)
if not results.empty:
    results["abs_ic_5d"] = results["ic_5d"].abs()
    results = results.sort_values("abs_ic_5d", ascending=False).reset_index(drop=True)

print("\n=== Ranked by |IC_5d| ===")
display_cols = [
    "factor",
    "ic_1d", "icir_1d",
    "ic_5d", "icir_5d",
    "ic_21d", "icir_21d",
]
print(results[display_cols].to_string(index=False, float_format=lambda x: f"{x: .4f}"))

print("\n=== Best signed IC_5d (top 10) ===")
print(
    results.sort_values("ic_5d", ascending=False)[display_cols]
    .head(10)
    .to_string(index=False, float_format=lambda x: f"{x: .4f}")
)

print("\n=== Worst signed IC_5d (bottom 10) ===")
print(
    results.sort_values("ic_5d", ascending=True)[display_cols]
    .head(10)
    .to_string(index=False, float_format=lambda x: f"{x: .4f}")
)


ok adx_10
ok adx_14
ok adx_20
ok adx_28
ok adx_7
ok bb_percent_b_10
ok bb_percent_b_15
ok bb_percent_b_20
ok bb_percent_b_30
ok bb_percent_b_40
ok bb_percent_b_50
ok mfi_10
ok mfi_14
ok mfi_20
ok mfi_28
ok mfi_7
ok rsi_10
ok rsi_14
ok rsi_2
ok rsi_21
ok rsi_28
ok rsi_5
ok rsi_7

=== Ranked by |IC_5d| ===
         factor   ic_1d  icir_1d   ic_5d  icir_5d  ic_21d  icir_21d
         rsi_28  0.0086   0.0407  0.0100   0.0460  0.0186    0.0883
bb_percent_b_50  0.0077   0.0387  0.0088   0.0434  0.0147    0.0765
bb_percent_b_10 -0.0061  -0.0328 -0.0087  -0.0484  0.0010    0.0056
         rsi_21  0.0065   0.0314  0.0079   0.0372  0.0166    0.0819
          rsi_2 -0.0068  -0.0369 -0.0060  -0.0341  0.0004    0.0025
bb_percent_b_40  0.0050   0.0257  0.0056   0.0283  0.0123    0.0658
         mfi_28  0.0055   0.0315  0.0053   0.0295  0.0083    0.0487
         adx_28  0.0030   0.0189  0.0051   0.0307  0.0019    0.0113
         rsi_14  0.0040   0.0198  0.0051   0.0251  0.0141    0.0730
bb_percent_b_1